# Summarize Private Documents Using RAG, LangChain & OpenAI

**Adapted from IBM Agentic AI Course — Section 2**

**Author:** Saif Ali Khan · 2026

---

## Overview

This notebook demonstrates **Retrieval-Augmented Generation (RAG)** — a technique that combines document retrieval with LLM-powered answer generation. Instead of relying solely on the LLM's training data, RAG retrieves relevant context from your documents and uses it to ground the answers, reducing hallucinations and improving accuracy.

### What You'll Learn
- What RAG is and why it matters
- How to load, chunk, and embed documents
- Building a vector store with ChromaDB
- Creating a RAG pipeline with LangChain
- Controlling LLM behavior with prompt templates
- Adding conversation memory for multi-turn dialogue
- Building an interactive chatbot

## Table of Contents

1. [Background — What is RAG?](#background)
2. [Setup — Install Libraries & API Key](#setup)
3. [Step 1: Load the Document](#step1)
4. [Step 2: Chunking](#step2)
5. [Step 3: Embedding + Vector Store (ChromaDB)](#step3)
6. [Step 4: Build the LLM](#step4)
7. [Step 5: Basic RAG — Ask Questions](#step5)
8. [Dive Deeper: Prompt Templates (Fixing Hallucinations)](#deeper1)
9. [Dive Deeper: Conversation Memory](#deeper2)
10. [Interactive Chatbot](#chatbot)
11. [Exercises](#exercises)
12. [Summary & Key Concepts](#summary)

## 1. Background — What is RAG?

### The Problem: Hallucinations

Large Language Models (LLMs) like GPT are trained on vast amounts of public internet data. When you ask a question, they generate responses based on patterns learned during training. **However, they have no knowledge of your private documents** — confidential policies, contracts, proprietary research, etc.

When asked about private documents, LLMs will confidently "invent" plausible-sounding answers. This is called **hallucination**.

### The Solution: RAG Pipeline

**Retrieval-Augmented Generation** adds a retrieval step before generation:

```
User Question
     |
     v
  [RETRIEVE] -> Search vector store for relevant chunks
     |
     v
  Retrieved context + Question
     |
     v
  [GENERATE] -> LLM reads context and answers
     |
     v
  Grounded Answer (less hallucination)
```

### Why RAG?

✓ **Accuracy**: Answers are grounded in your actual documents

✓ **Privacy**: Documents stay on your system (not in LLM training)

✓ **Control**: You can update documents without retraining

✓ **Cost**: Cheaper than fine-tuning models

## 2. Setup — Install Libraries & API Key

We'll use:
- **LangChain**: Framework for building RAG pipelines
- **OpenAI API**: Access to GPT-4o-mini LLM and embeddings
- **ChromaDB**: Vector database to store & search embeddings
- **python-dotenv**: Load API keys from `.env` file

Make sure you have a `.env` file in your working directory with:
```
OPENAI_API_KEY=sk-proj-...
```

In [ ]:
# Remove any conflicting v1.x packages that might cause import errors
!pip uninstall -y langchain-classic langchain-experimental langgraph-prebuilt 2>nul

# Install compatible packages for LangChain 0.3+
# We specify exact version ranges to avoid breaking changes
!pip install -q "langchain>=0.3.0,<0.4.0" \
    "langchain-openai>=0.2.0,<0.3.0" \
    "langchain-community>=0.3.0,<0.4.0" \
    "langchain-core>=0.3.0,<0.4.0" \
    "langchain-text-splitters>=0.3.0,<0.4.0" \
    "chromadb>=0.5.5" \
    python-dotenv \
    requests

print("✓ All packages installed successfully!")

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Create a cache directory for tiktoken (OpenAI's tokenizer)
# This prevents DNS issues when running offline
cache_dir = os.path.join(os.getcwd(), 'tiktoken_cache')
os.makedirs(cache_dir, exist_ok=True)
os.environ['TIKTOKEN_CACHE_DIR'] = cache_dir

# Retrieve the API key
api_key = os.getenv("OPENAI_API_KEY")

# Validate that the key looks correct
if api_key and api_key.startswith("sk-"):
    print("✓ API key loaded successfully!")
else:
    print("⚠️ WARNING: API key not found or looks wrong.")
    print("Make sure you have a .env file with: OPENAI_API_KEY=sk-proj-...")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Document loading
from langchain_community.document_loaders import TextLoader

# Text splitting / chunking
from langchain_text_splitters import CharacterTextSplitter

# Embeddings (convert text to vectors)
from langchain_openai import OpenAIEmbeddings

# Vector store (search & retrieve)
from langchain_community.vectorstores import Chroma

# Language model
from langchain_openai import ChatOpenAI

# RAG chains
from langchain.chains import RetrievalQA, ConversationalRetrievalChain

# Prompt template (control LLM behavior)
from langchain.prompts import PromptTemplate

# Memory (track conversation history)
from langchain.memory import ConversationBufferMemory

import requests

print("✓ All imports successful!")

## 3. Step 1: Load the Document

The RAG pipeline starts with loading your document(s). In this example, we'll load a local text file containing company policies.

**Why TextLoader?** It's simple and works for plain text files. For other formats (PDF, Word, etc.), you'd use different loaders like `PyPDFLoader` or `UnstructuredPDFLoader`.

In [ ]:
# Specify the filename to load
filename = "companyPolicies.txt"

# Load the file with UTF-8 encoding (handles special characters)
with open(filename, 'r', encoding='utf-8') as f:
    contents = f.read()

# Display basic info about the loaded document
print(f"✓ File loaded: {filename}")
print(f"  File size: {len(contents)} characters")
print(f"  File size: {len(contents.split())} words")

In [ ]:
# Read and display the document contents
with open(filename, "r", encoding='utf-8') as file:
    contents = file.read()

# Show the first 2000 characters
print(contents[:2000])
print("\n... (truncated for display) ...")
print(f"\nTotal length: {len(contents)} characters")

## 4. Step 2: Chunking

**Why chunk?** LLMs have a maximum context window (e.g., 8k tokens). Large documents don't fit, and even if they did, the LLM performs better with focused context.

**Chunking strategy:**
- **chunk_size=500**: Each chunk is roughly 500 characters
- **chunk_overlap=50**: Chunks overlap by 50 characters (preserves context across boundaries)
- **separator="\\n"**: Split on newlines first (preserves logical paragraphs)

This produces a list of smaller, overlapping text pieces that we'll embed and index.

In [ ]:
# Load document using LangChain's TextLoader
loader = TextLoader(filename, encoding='utf-8')
documents = loader.load()

# Initialize the text splitter with our chunking parameters
text_splitter = CharacterTextSplitter(
    chunk_size=500,        # Each chunk is ~500 characters
    chunk_overlap=50,      # 50 char overlap between chunks
    separator="\n"        # Split on newlines first
)

# Split documents into chunks
texts = text_splitter.split_documents(documents)

# Display results
print(f"Original document: 1 file")
print(f"After chunking: {len(texts)} chunks")
print(f"\n--- Example: Chunk #0 ---")
print(texts[0].page_content)
print(f"\n--- Example: Chunk #3 ---")
print(texts[3].page_content)

## 5. Step 3: Embedding + Vector Store (ChromaDB)

**Embedding** converts text into high-dimensional vectors (lists of numbers). Semantically similar texts have similar vectors.

- OpenAI's embedding model creates 1536-dimensional vectors
- We store these in **ChromaDB**, a vector database
- ChromaDB enables fast similarity search ("What chunks are relevant to this question?")

**The key insight:** When you ask a question, we:
1. Embed your question
2. Search for chunks with similar embeddings
3. Pass those chunks to the LLM as context

In [ ]:
# Initialize the embedding model
# OpenAI's text-embedding-3-small model creates 1536-dimensional vectors
embeddings = OpenAIEmbeddings()

# Create a vector store from the chunks
# This embeds all chunks and stores them in ChromaDB
docsearch = Chroma.from_documents(texts, embeddings)

# Display confirmation
print(f"✓ Document ingested!")
print(f"  ChromaDB now contains {len(texts)} chunk vectors")
print(f"  Each vector has 1536 dimensions (numbers)")
print(f"  This enables fast semantic search across your document")

In [ ]:
# Let's see what an embedding actually looks like
# Embed a sample query
sample_vector = embeddings.embed_query("What is the mobile phone policy?")

# Display vector information
print(f"Input text: 'What is the mobile phone policy?'")
print(f"Output vector length: {len(sample_vector)} numbers")
print(f"\nFirst 10 numbers: {sample_vector[:10]}")
print(f"\n^ Each number captures some aspect of the text's meaning.")
print(f"  Similar questions will produce similar vectors.")
print(f"  This is how ChromaDB finds relevant chunks!")

## 6. Step 4: Build the LLM

Now we initialize the **Language Model** that will generate answers.

We're using **ChatOpenAI** with:
- **model="gpt-4o-mini"**: Fast, cost-effective variant of GPT-4
- **temperature=0.3**: Lower temperature = more focused, deterministic answers (good for factual Q&A)

In [ ]:
# Initialize the language model
llm = ChatOpenAI(
    model="gpt-4o-mini",  # Lightweight version of GPT-4
    temperature=0.3       # Lower temp = more focused answers (0.0-1.0)
)

print("✓ LLM ready: GPT-4o-mini (temperature=0.3)")
print("  This model will generate answers based on retrieved context.")

## 7. Step 5: Basic RAG — Ask Questions

Now we have all the pieces:
- ✓ Embedded document chunks in ChromaDB
- ✓ Language model ready to generate answers

We combine them into a **RAG chain** using LangChain's `RetrievalQA`:

```
Question → Retrieve relevant chunks → Pass to LLM → Answer
```

**chain_type="stuff"** means we "stuff" all retrieved context into the LLM's context window. (Other strategies include "map_reduce" for very large contexts.)

In [ ]:
# Create a RetrievalQA chain
# This combines retrieval + generation into one pipeline
qa = RetrievalQA.from_chain_type(
    llm=llm,                           # The language model
    chain_type="stuff",               # Strategy: stuff all context into one prompt
    retriever=docsearch.as_retriever(), # Retrieval from ChromaDB
    return_source_documents=False      # Don't return the source chunks
)

# Ask a question
query = "What is the mobile phone policy?"
result = qa.invoke(query)

# Display the answer
print(f"Question: {query}")
print(f"\nAnswer: {result['result']}")

In [ ]:
# Test: Can it summarize the document?
query = "Can you summarize the document for me?"
result = qa.invoke(query)

print(f"Question: {query}")
print(f"\nAnswer: {result['result']}")

In [ ]:
# Test: Does the LLM hallucinate when asked about something NOT in the document?
query = "Can I eat in company vehicles?"
result = qa.invoke(query)

print(f"Question: {query}")
print(f"\nAnswer: {result['result']}")
print(f"\n⚠️ Hallucination Check:")
print(f"   The document talks about SMOKING in vehicles, not EATING.")
print(f"   If the LLM gave an answer about eating, it hallucinated!")
print(f"   (This shows the need for better prompts → next section)")

## 8. Dive Deeper: Prompt Templates (Fixing Hallucinations)

The basic RAG chain sometimes hallucinates because the default prompt doesn't explicitly tell the LLM to **stay within the document**.

**Solution: Create a custom prompt template** that:
1. Provides retrieved context
2. Asks the question
3. **Instructs the LLM to only answer from the context**
4. Tells it to say "I don't know" if the answer isn't in the document

This dramatically reduces hallucinations!

In [ ]:
# Define a custom prompt template that prevents hallucinations
prompt_template = """Use the following context from the document to answer the question at the end.
If you don't know the answer based on the context, just say "I don't know based on the provided document."
Do NOT make up an answer.

Context:
{context}

Question: {question}

Answer:"""

# Create a PromptTemplate object
PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]  # Variables that will be filled in
)

# Prepare to pass the custom prompt to the chain
chain_type_kwargs = {"prompt": PROMPT}

print("✓ Custom prompt template created!")
print("  The LLM will now be instructed to only answer from the document.")

In [ ]:
# Create a new RAG chain with the custom prompt
qa_with_prompt = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    chain_type_kwargs=chain_type_kwargs,  # Pass the custom prompt
    return_source_documents=False
)

# Test: Ask about something NOT in the document
query = "Can I eat in company vehicles?"
result = qa_with_prompt.invoke(query)

print(f"Question: {query}")
print(f"\nAnswer: {result['result']}")
print(f"\n✓ Notice: The LLM now says it doesn't know, instead of hallucinating!")

In [ ]:
# Test: Ask about something that IS in the document
query = "What is the smoking policy?"
result = qa_with_prompt.invoke(query)

print(f"Question: {query}")
print(f"\nAnswer: {result['result']}")
print(f"\n✓ When the information IS in the document, the LLM provides it!")

In [ ]:
# Test: Multi-turn dialogue without memory
# Ask a follow-up question that depends on context from the previous question
query = "What can't I do in it?"
result = qa_with_prompt.invoke(query)

print(f"Question: {query}")
print(f"\nAnswer: {result['result']}")
print(f"\n⚠️ Problem: The LLM doesn't remember the previous question about smoking!")
print(f"   'it' has no context. We need MEMORY for multi-turn conversations.")

## 9. Dive Deeper: Conversation Memory

The basic RAG chain doesn't remember previous questions. Each query is treated independently.

**For chatbot-like interactions, we need memory:**
- Store the conversation history
- Include it in the next prompt
- Enable natural multi-turn dialogue

We'll use `ConversationBufferMemory` (stores the full history) and `ConversationalRetrievalChain` (RAG + memory).

In [ ]:
# Initialize conversation memory
# This will store all turns of the conversation
memory = ConversationBufferMemory(
    memory_key="chat_history",  # Key to store conversation in the prompt
    return_messages=True          # Return as Message objects (for better formatting)
)

print("✓ Memory initialized — conversation history will be tracked.")

In [ ]:
# Create a ConversationalRetrievalChain
# This combines RAG + memory for multi-turn dialogue
qa_with_memory = ConversationalRetrievalChain.from_llm(
    llm=llm,                              # Language model
    chain_type="stuff",                  # Strategy
    retriever=docsearch.as_retriever(),   # Retrieve from vector store
    memory=memory,                        # Include conversation history
    return_source_documents=False         # Don't return source chunks
)

print("✓ Conversational chain ready — now with memory!")

In [ ]:
# Turn 1: Ask about the mobile phone policy
query = "What is the mobile phone policy?"
result = qa_with_memory.invoke({"question": query})

print(f"You: {query}")
print(f"\nBot: {result['answer']}")

In [ ]:
# Turn 2: Follow-up with a pronoun "it"
# The memory helps the LLM understand "it" refers to the mobile phone policy
query = "List all the points in it."
result = qa_with_memory.invoke({"question": query})

print(f"You: {query}")
print(f"\nBot: {result['answer']}")
print(f"\n✓ Notice: The LLM understands 'it' = mobile phone policy (from memory)")

In [ ]:
# Turn 3: Another follow-up
query = "What is the aim of it?"
result = qa_with_memory.invoke({"question": query})

print(f"You: {query}")
print(f"\nBot: {result['answer']}")
print(f"\n✓ Memory enables natural, context-aware conversations!")

## 10. Interactive Chatbot

Now let's build an **interactive chatbot** that:
- Takes user input in a loop
- Retrieves relevant chunks from the vector store
- Generates responses using the LLM
- Remembers the conversation history

This is a complete RAG chatbot! In production, you'd wrap this in a web app (Streamlit, Flask, etc.).

In [ ]:
def chatbot():
    """
    Interactive RAG chatbot for company policy questions.
    
    Workflow:
    1. User enters a question
    2. ChromaDB retrieves relevant chunks
    3. LLM generates an answer from those chunks
    4. Memory stores the conversation
    5. Repeat for next question
    """
    
    # Initialize memory for this session
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )
    
    # Create the conversational chain
    qa = ConversationalRetrievalChain.from_llm(
        llm=llm,
        chain_type="stuff",
        retriever=docsearch.as_retriever(),
        memory=memory,
        return_source_documents=False
    )
    
    # Display welcome message
    print("=" * 60)
    print("  RAG CHATBOT — Ask anything about the company policies!")
    print("  Type 'quit', 'exit', or 'bye' to stop.")
    print("=" * 60)
    
    # Main conversation loop
    while True:
        # Get user input
        query = input("\nYou: ")
        
        # Check for exit commands
        if query.lower().strip() in ["quit", "exit", "bye", "q"]:
            print("\nBot: Goodbye!")
            break
        
        # Skip empty input
        if not query.strip():
            continue
        
        # Generate and display response
        result = qa.invoke({"question": query})
        print(f"\nBot: {result['answer']}")

print("✓ Chatbot function defined!")
print("  Run the next cell to start the interactive session.")

In [ ]:
# Uncomment the line below to run the interactive chatbot
# chatbot()

# Note: In Jupyter notebooks, interactive input() doesn't always work well.
# For a production chatbot, use Streamlit, Flask, or Discord bots instead.
print("✓ To run the chatbot, uncomment the chatbot() line above.")
print("  For production, consider Streamlit or Flask web apps.")

## 11. Exercises

Try these to deepen your understanding:

### Exercise 1: Use Your Own Document
Load a different text file (e.g., a research paper, handbook, or policy document) and build a RAG pipeline on top of it. What types of questions does it answer well? Where does it hallucinate?

In [ ]:
# TODO: Load your own document
# Hints:
# 1. Change filename to your file
# 2. Follow the same steps: Load → Chunk → Embed → Build RAG Chain
# 3. Test with questions specific to your document

# your_filename = "your_document.txt"
# ... (insert code here)

pass

### Exercise 2: Return Source Documents
Modify the RAG chain to return the source chunks it used to generate the answer. This helps users verify that answers are grounded in the document.

In [ ]:
# TODO: Rebuild the RAG chain with return_source_documents=True
# Then ask a question and display:
# 1. The answer
# 2. The source chunks that were retrieved
# 3. Why those chunks were relevant

# Hint: Change return_source_documents to True in RetrievalQA.from_chain_type()
# Then access result['source_documents'] to see the chunks

pass

### Exercise 3: Try a Different Model
Test with a different OpenAI model (e.g., "gpt-4", "gpt-3.5-turbo"). How do results differ? Is the quality worth the cost?

In [ ]:
# TODO: Create a new LLM with a different model
# Test the same questions and compare results

# alternative_llm = ChatOpenAI(model="gpt-4", temperature=0.3)
# ... (build RAG chain and test)

pass

### Exercise 4: Experiment with Chunking Parameters
How do chunk_size and chunk_overlap affect results? Try different values and observe:

In [ ]:
# TODO: Test different chunking strategies
# 1. Small chunks (200 chars) vs. large chunks (1000 chars)
# 2. No overlap vs. high overlap (100 chars)
# 3. Different separators ("\\n\\n" vs "." vs "\\n")
# 
# Then rebuild the vector store and compare retrieved chunks.
# Which strategy retrieved the most relevant chunks?

pass

## 12. Summary & Key Concepts

Congratulations! You've built a complete **RAG pipeline** from scratch. Here's what you learned:

### RAG Pipeline Components

| Concept | What it does | LangChain Component | Why it matters |
|---------|-------------|-------------------|----------------|
| **Chunking** | Split documents into pieces | CharacterTextSplitter | LLM context window is limited |
| **Embedding** | Convert text to vectors | OpenAIEmbeddings | Enables semantic similarity search |
| **Vector Store** | Store & search vectors | Chroma | Fast retrieval of relevant chunks |
| **LLM** | Generate answers | ChatOpenAI | Uses context to answer questions |
| **Retrieval Chain** | Question → Retrieve → Answer | RetrievalQA | Combines retrieval + generation |
| **Prompt Template** | Control LLM behavior | PromptTemplate | Prevents hallucinations |
| **Memory** | Remember conversation | ConversationBufferMemory | Enables multi-turn dialogue |
| **Conversational RAG** | RAG + Memory | ConversationalRetrievalChain | Chatbot-like interactions |

### The RAG Workflow

**Setup Phase:**
1. Load your documents (PDF, TXT, etc.)
2. Split into overlapping chunks (500 chars)
3. Convert each chunk to 1536-dimensional vectors
4. Store vectors in ChromaDB (vector database)

**Query Phase (repeats for each question):**
1. Embed the question
2. Search ChromaDB for similar chunks
3. Pass retrieved chunks + question to LLM
4. LLM reads context and generates answer
5. Return grounded answer (less hallucination)

### Key Insights

1. **RAG solves hallucination**: By retrieving context from your actual documents, the LLM has ground truth to anchor answers.

2. **Embeddings enable semantic search**: Instead of keyword matching, embeddings capture meaning. "What's the phone policy?" matches chunks about mobile phone usage.

3. **Prompt engineering matters**: A well-designed prompt dramatically improves results. Tell the LLM to say "I don't know" instead of hallucinating.

4. **Memory enables conversations**: Without memory, each query is independent. With memory, the chatbot understands pronouns and context from earlier turns.

5. **Chunking is a hyperparameter**: Chunk size, overlap, and separator affect what gets retrieved. Small chunks are more precise; large chunks preserve context.

6. **Cost and quality trade-offs**: GPT-4o-mini is fast and cheap. GPT-4 is slower and pricier but higher quality. Choose based on your use case.

7. **Observability is crucial**: Always return source documents in production so users can verify that answers are grounded in the document.

### IBM to OpenAI Mapping

This notebook was adapted from the IBM Agentic AI course. Here's what changed:

| IBM Component | OpenAI Alternative | Why |
|---------------|-------------------|-----|
| IBM Watson NLU | OpenAI Embeddings | Text to vectors for similarity search |
| Db2 Vector Store | ChromaDB | Store & retrieve high-dimensional vectors |
| IBM Watsonx | ChatOpenAI (GPT-4o-mini) | Generate answers from context |
| IBM Watson Assistant | ConversationalRetrievalChain | Build chatbots with memory |

Both approaches follow the same RAG pattern. The specific tools differ, but the concepts are universal.

### Next Steps

Want to take RAG further? Try these:

- **Use case**: Build a RAG app for your team's internal documents (SOPs, onboarding, FAQs)
- **Optimize retrieval**: Experiment with different embedding models, chunk sizes, and retrieval strategies
- **Add multi-document support**: Extend to search across multiple files or folders
- **Deploy as web app**: Use Streamlit or Flask to build a user-friendly interface
- **Monitor performance**: Track which queries hallucinate most; refine prompts accordingly
- **Hybrid retrieval**: Combine vector search with keyword/BM25 search for robustness
- **Advanced memory**: Try ConversationSummaryMemory or ConversationKGMemory for long conversations

---

**Happy building!**